# K3A × Sentinel-2 칩 생성 파이프라인 (v9-B)

## 처리 내용
1. 공통 설정 + ortho_result_map 로드
2. 유틸리티 함수
3. 파이프라인 실행

## Phase Correlation 순서 (핵심)
```
[씬 단위]
K3A ortho R밴드 로드
S2 R밴드를 K3A 격자(2.5m)로 업샘플링
Phase Correlation → (dy_px, dx_px) 추정
    ↓
S2 스택 전체 transform origin 이동 → s2_shifted.tif 저장
(픽셀값 재보간 없음 — transform만 수정)
    ↓
[칩 단위]
shift된 S2로 칩 샘플링 (K3A/S2 동일 origin 사용)
```
→ 노트북 A의 `ortho_result_map.json` 필요


## 0. 설치 및 공통 설정

In [ ]:
# ======================================================================
# 경로 설정 — 본인 환경에 맞게 수정하세요
# ======================================================================
import os

# Colab을 쓰는 경우에만 drive mount
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DEFAULT_ROOT = '/content/drive/MyDrive/kompsat_sr'  # ← 본인 드라이브 경로로 수정
except ImportError:
    DEFAULT_ROOT = './data'  # 로컬 환경

PROJECT_ROOT     = os.environ.get('PROJECT_ROOT', DEFAULT_ROOT)
DATASET_ROOT     = os.path.join(PROJECT_ROOT, 'dataset')
CHECKPOINT_DIR   = os.path.join(PROJECT_ROOT, 'checkpoints')
PRITHVI_DIR      = os.path.join(PROJECT_ROOT, 'prithvi')
SR_ROOT          = PROJECT_ROOT
VAL_DATASET_ROOT = os.path.join(PROJECT_ROOT, 'val_dataset')

os.makedirs(PROJECT_ROOT, exist_ok=True)
print(f'✅ PROJECT_ROOT = {PROJECT_ROOT}')


In [ ]:
!pip install -q rasterio shapely tqdm scikit-image
!apt-get install -q -y gdal-bin python3-gdal

import os, glob, zipfile, json, csv, warnings, gc
import numpy as np
import rasterio
from rasterio.warp import reproject, Resampling, transform_bounds
from rasterio.transform import Affine
from skimage.registration import phase_cross_correlation
from scipy.ndimage import shift as scipy_shift
from shapely.geometry import box
from tqdm.notebook import tqdm
warnings.filterwarnings('ignore')

BASE_DIR     = f'{PROJECT_ROOT}'
DOWNLOAD_DIR = os.path.join(BASE_DIR, 'download_data')
EXTRACT_DIR  = os.path.join(BASE_DIR, 'extracted')
ORTHO_DIR    = os.path.join(BASE_DIR, 'k3a_ortho')
CHIP_DIR     = os.path.join(BASE_DIR, 'generated_chip')
ORTHO_MAP_JSON = os.path.join(BASE_DIR, 'ortho_result_map.json')

K3A_CHIP = 896;  S2_CHIP = 224
K3A_RES  = 2.5;  S2_RES  = 10.0
K3A_NODATA_THRESH = 0.02
S2_NODATA_THRESH  = 0.02

S2_BANDS  = {'R': 'B04', 'G': 'B03', 'B': 'B02', 'NIR': 'B08'}

for d in [CHIP_DIR]: os.makedirs(d, exist_ok=True)

# ── ortho_result_map 로드 ──────────────────────────────────────
if not os.path.exists(ORTHO_MAP_JSON):
    raise FileNotFoundError(f'노트북 A를 먼저 실행하세요: {ORTHO_MAP_JSON}')

with open(ORTHO_MAP_JSON) as f:
    ortho_result_map = json.load(f)

print(f'ortho_result_map 로드 완료: {len(ortho_result_map)}개 씬')
print('환경 설정 완료')


## 1. 유틸리티 함수

In [ ]:
def extract_s2_zip(zip_path, extract_dir):
    with zipfile.ZipFile(zip_path, 'r') as z:
        safe_name = next((n.split('/')[0] for n in z.namelist() if '.SAFE' in n), None)
    if not safe_name: raise RuntimeError(f'.SAFE 없음: {zip_path}')
    safe_path = os.path.join(extract_dir, safe_name)
    if not os.path.exists(safe_path):
        with zipfile.ZipFile(zip_path, 'r') as z: z.extractall(extract_dir)
    return safe_path

def find_s2_band_file(safe_root, band_name):
    for pat in [
        f'GRANULE/*/IMG_DATA/R10m/*_{band_name}_10m.jp2',
        f'GRANULE/*/IMG_DATA/R10m/*_{band_name}_10m.tif',
        f'GRANULE/*/IMG_DATA/*_{band_name}.jp2',
        f'GRANULE/*/IMG_DATA/*_{band_name}.tif']:
        hits = glob.glob(os.path.join(safe_root, pat))
        if hits: return hits[0]
    return None

def stack_s2_bands(safe_root, out_path):
    """S2 R/G/B/NIR 밴드를 4밴드 GeoTIFF로 스택. 이미 있으면 스킵."""
    # 이미 존재하고 크기가 충분하면 무결성 체크
    if os.path.exists(out_path) and os.path.getsize(out_path) > 500*1024*1024:
        # 실제로 열릴 수 있는지 확인
        try:
            with rasterio.open(out_path) as src:
                src.read(1, window=rasterio.windows.Window(0, 0, 10, 10))
            return out_path  # 정상
        except Exception:
            print(f'  ⚠️  손상된 스택 감지 → 재생성: {os.path.basename(out_path)}')
            os.remove(out_path)

    if os.path.exists(out_path):
        os.remove(out_path)

    band_files = {k: find_s2_band_file(safe_root, v) for k, v in S2_BANDS.items()}
    missing = [k for k, v in band_files.items() if v is None]
    if missing: raise FileNotFoundError(f'S2 밴드 없음: {missing}')
    with rasterio.open(band_files['R']) as ref:
        meta = ref.meta.copy()
        meta.update(count=4, driver='GTiff', compress='lzw')
    with rasterio.open(out_path, 'w', **meta) as dst:
        for i, key in enumerate(['R', 'G', 'B', 'NIR'], start=1):
            with rasterio.open(band_files[key]) as src: dst.write(src.read(1), i)
    return out_path


def estimate_scene_shift(k3a_ortho_path, s2_stack_path):
    from skimage.registration import phase_cross_correlation

    # 실제 경로 확인
    print(f'    k3a: {os.path.basename(k3a_ortho_path)}')
    print(f'    s2:  {os.path.basename(s2_stack_path)}')

    with rasterio.open(k3a_ortho_path) as src:
        k3a_tf  = src.transform
        k3a_crs = src.crs

    with rasterio.open(s2_stack_path) as src:
        s2_r     = src.read(1).astype(np.float32)
        s2_tf    = src.transform
        s2_crs   = src.crs
        s2_shape = (src.height, src.width)
        print(f'    s2_r max={s2_r.max():.0f}  >0={( s2_r>0).mean():.1%}')

    k3a_down = np.zeros(s2_shape, dtype=np.float32)
    with rasterio.open(k3a_ortho_path) as src:
        reproject(source=rasterio.band(src, 1),
                  destination=k3a_down,
                  src_transform=k3a_tf, src_crs=k3a_crs,
                  dst_transform=s2_tf,  dst_crs=s2_crs,
                  resampling=Resampling.average)

    print(f'    k3a_down max={k3a_down.max():.0f}  >0={(k3a_down>0).mean():.2%}')

    valid   = (k3a_down > 0) & (s2_r > 0)
    n_valid = valid.sum()
    print(f'    교집합 유효 픽셀: {n_valid:,}개 ({valid.mean():.2%} of S2 tile)')

    if n_valid < 100:
        print('    ⚠️  유효 픽셀 부족 → shift=0')
        del k3a_down, s2_r; gc.collect()
        return 0.0, 0.0

    rows = np.any(valid, axis=1)
    cols = np.any(valid, axis=0)
    r0, r1 = np.where(rows)[0][[0, -1]]
    c0, c1 = np.where(cols)[0][[0, -1]]

    shift, *_ = phase_cross_correlation(
        k3a_down[r0:r1, c0:c1], s2_r[r0:r1, c0:c1],
        upsample_factor=20, normalization='phase')
    dy_s2, dx_s2 = float(shift[0]), float(shift[1])

    dx_px = dx_s2 * (S2_RES / K3A_RES)
    dy_px = dy_s2 * (S2_RES / K3A_RES)

    MAX_SHIFT_PX = 200
    if abs(dx_px) > MAX_SHIFT_PX or abs(dy_px) > MAX_SHIFT_PX:
        print(f'    ⚠️  shift 과대 → 0 fallback')
        del k3a_down, s2_r; gc.collect()
        return 0.0, 0.0

    del k3a_down, s2_r; gc.collect()
    return dx_px, dy_px


def apply_scene_shift(s2_stack_path, out_path, dx_px, dy_px):
    """
    S2 스택 전체를 shift 적용해서 새 파일로 저장.

    픽셀값 재보간 없음 — transform origin만 수정.
    dx_px, dy_px는 K3A 2.5m 픽셀 기준이므로 미터로 변환.

    부호 규칙:
        Phase Corr이 'S2가 K3A보다 dx_px만큼 오른쪽에 있다'고 추정하면
        S2 origin을 dx_px × 2.5m 왼쪽으로 이동 → c - shift_x
        y축은 래스터 좌표계에서 위가 음수이므로 f + shift_y
    """
    if os.path.exists(out_path):
        return out_path

    shift_x_m = dx_px * K3A_RES   # 미터
    shift_y_m = dy_px * K3A_RES

    with rasterio.open(s2_stack_path) as src:
        data   = src.read()
        meta   = src.meta.copy()
        old_tf = src.transform

    # transform origin에 shift 반영
    # c: 좌상단 x (동쪽 +), f: 좌상단 y (북쪽 +)
    new_tf = Affine(
        old_tf.a, old_tf.b, old_tf.c + shift_x_m,
        old_tf.d, old_tf.e, old_tf.f - shift_y_m,
    )
    meta.update(transform=new_tf, compress='lzw')

    with rasterio.open(out_path, 'w', **meta) as dst:
        dst.write(data)

    return out_path


def make_chip_grid(k3a_bounds, k3a_crs, s2_stack_path, k3a_chip_sz=896, k3a_res=2.5):
    """교집합 중심 기반 칩 그리드 생성. snap 없음."""
    with rasterio.open(s2_stack_path) as s2:
        common_crs = s2.crs
        s2_bounds  = s2.bounds
    k3a_left, k3a_bottom, k3a_right, k3a_top = transform_bounds(
        k3a_crs, common_crs,
        k3a_bounds.left, k3a_bounds.bottom,
        k3a_bounds.right, k3a_bounds.top)
    inter_left   = max(k3a_left,   s2_bounds.left)
    inter_bottom = max(k3a_bottom, s2_bounds.bottom)
    inter_right  = min(k3a_right,  s2_bounds.right)
    inter_top    = min(k3a_top,    s2_bounds.top)
    if inter_left >= inter_right or inter_bottom >= inter_top:
        raise RuntimeError('K3A-S2 교집합 없음')
    chip_m      = k3a_chip_sz * k3a_res
    cx          = (inter_left + inter_right)  / 2
    cy          = (inter_bottom + inter_top)  / 2
    half_cols   = int((inter_right  - inter_left)  / chip_m / 2)
    half_rows   = int((inter_top    - inter_bottom) / chip_m / 2)
    origin_left = cx - chip_m / 2
    origin_top  = cy + chip_m / 2
    chips = []
    for di in range(-half_rows, half_rows + 1):
        for dj in range(-half_cols, half_cols + 1):
            cl = origin_left + dj * chip_m
            ct = origin_top  - di * chip_m
            if (cl >= inter_left and cl + chip_m <= inter_right and
                ct - chip_m >= inter_bottom and ct <= inter_top):
                chips.append((cl, ct))
    return chips, common_crs


def sample_chip(bands, src_transform, src_crs, common_crs,
                chip_left, chip_top, chip_sz, res):
    """메모리에 올려둔 밴드에서 칩 영역 bilinear 샘플링."""
    chip_tf   = Affine(res, 0, chip_left, 0, -res, chip_top)
    bands_out = []
    for key in ['R', 'G', 'B', 'NIR']:
        dst = np.zeros((chip_sz, chip_sz), dtype=np.float32)
        reproject(source=bands[key], destination=dst,
                  src_transform=src_transform, src_crs=src_crs,
                  dst_transform=chip_tf,       dst_crs=common_crs,
                  resampling=Resampling.bilinear)
        bands_out.append(dst)
    return np.stack(bands_out, axis=0)


def chip_pair_v9(k3a_ortho_path, s2_stack_path, chip_dir, pair_name,
                 k3a_chip_sz=896, s2_chip_sz=224,
                 k3a_res=2.5, s2_res=10.0,
                 k3a_nodata_thresh=0.02, s2_nodata_thresh=0.02):
    """
    칩 생성 메인 함수.
    s2_stack_path는 이미 shift 적용된 파일을 받음.
    K3A와 S2 모두 동일한 origin 사용 — 별도 offset 불필요.
    """
    os.makedirs(chip_dir, exist_ok=True)

    keys = ['R', 'G', 'B', 'NIR']
    with rasterio.open(k3a_ortho_path) as src:
        k3a_tf     = src.transform
        k3a_crs    = src.crs
        k3a_bounds = src.bounds
        k3a_bands  = {keys[i]: src.read(i+1).astype(np.float32) for i in range(4)}

    with rasterio.open(s2_stack_path) as src:
        s2_tf    = src.transform
        s2_crs   = src.crs
        s2_bands = {keys[i]: src.read(i+1).astype(np.float32) for i in range(4)}

    chips, common_crs = make_chip_grid(
        k3a_bounds, k3a_crs, s2_stack_path, k3a_chip_sz, k3a_res)
    print(f'     후보 칩: {len(chips)}개')
    if len(chips) == 0:
        print('     ⚠️  교집합이 칩 1장보다 작음 — 스킵')
        return 0, 0

    n_saved, n_drop = 0, 0
    meta_rows = []
    base_meta = {'driver': 'GTiff', 'dtype': 'float32', 'nodata': 0,
                 'count': 4, 'crs': common_crs, 'compress': 'lzw'}

    for cl, ct in chips:
        k_chip = sample_chip(k3a_bands, k3a_tf, k3a_crs, common_crs,
                             cl, ct, k3a_chip_sz, k3a_res)
        s_chip = sample_chip(s2_bands,  s2_tf,  s2_crs,  common_crs,
                             cl, ct, s2_chip_sz, s2_res)

        k_nd = (k_chip == 0).mean()
        s_nd = (s_chip == 0).mean()
        if k_nd > k3a_nodata_thresh or s_nd > s2_nodata_thresh:
            n_drop += 1; continue

        fname       = f'{pair_name}_{n_saved:04d}'
        k3a_chip_tf = Affine(k3a_res, 0, cl, 0, -k3a_res, ct)
        s2_chip_tf  = Affine(s2_res,  0, cl, 0, -s2_res,  ct)

        with rasterio.open(os.path.join(chip_dir, f'{fname}_K3A.tif'),
                           'w', width=k3a_chip_sz, height=k3a_chip_sz,
                           transform=k3a_chip_tf, **base_meta) as dst:
            dst.write(k_chip)
        with rasterio.open(os.path.join(chip_dir, f'{fname}_S2.tif'),
                           'w', width=s2_chip_sz, height=s2_chip_sz,
                           transform=s2_chip_tf, **base_meta) as dst:
            dst.write(s_chip)

        meta_rows.append({
            'fname':           fname,
            'left':            round(cl, 2),
            'top':             round(ct, 2),
            'k3a_nodata_pct':  round(k_nd*100, 2),
            's2_nodata_pct':   round(s_nd*100, 2),
        })
        n_saved += 1

    if meta_rows:
        csv_path = os.path.join(chip_dir, f'{pair_name}_chip_meta.csv')
        with open(csv_path, 'w', newline='') as f:
            writer = csv.DictWriter(f, fieldnames=meta_rows[0].keys())
            writer.writeheader(); writer.writerows(meta_rows)

    return n_saved, n_drop


print('✅ 유틸리티 함수 정의 완료')


## 2. 파이프라인 실행

### 순서
```
[1] S2 밴드 스택
[2] 씬 단위 Phase Corr → S2 shift 파일 생성
[3] 칩 생성 (shift된 S2 사용)
```


In [ ]:
import os

# L2A 스택 삭제
bad_stack = 'PROJECT_ROOT}/extracted/S2A_MSIL2A_20170304T021601_N0500_R003_T52SCG_20230926T182425_S2_stack.tif'
if os.path.exists(bad_stack):
    os.remove(bad_stack)
    print(f'삭제: {os.path.basename(bad_stack)}')

In [ ]:
region_folders = sorted(
    f for f in os.listdir(BASE_DIR)
    if os.path.isdir(os.path.join(BASE_DIR, f)) and f[0].isdigit())

pairs = []
for region in region_folders:
    region_path = os.path.join(BASE_DIR, region)
    for k3a_folder in sorted(os.listdir(region_path)):
        if not k3a_folder.startswith('K3A'): continue
        if k3a_folder not in ortho_result_map: continue
        # 수정: L1C 우선, 없으면 L2A
        matched_zips = glob.glob(os.path.join(DOWNLOAD_DIR, f'{k3a_folder}_*MSIL1C*.zip'))
        if not matched_zips:
            matched_zips = glob.glob(os.path.join(DOWNLOAD_DIR, f'{k3a_folder}_*MSIL2A*.zip'))
        if matched_zips:
            pairs.append((
                k3a_folder,
                ortho_result_map[k3a_folder],
                matched_zips[0]
            ))

print(f'처리 대상: {len(pairs)}개 씬')

results_log = []

for k3a_folder, k3a_ortho_path, s2_zip_path in pairs:
    print(f"\n{'='*60}\n🔄 {k3a_folder}")

    # 이미 저장된 칩 확인 → 전부 있으면 스킵
    existing_chips = glob.glob(os.path.join(CHIP_DIR, f'{k3a_folder}_*_K3A.tif'))
    if existing_chips:
        print(f'  ℹ️  이미 {len(existing_chips)}칩 존재 → 스킵')
        results_log.append((k3a_folder, len(existing_chips), 0, 0.0, 0.0))
        continue

    try:
        # Step 1: S2 밴드 스택
        print('  [1/3] S2 밴드 스택')
        safe_root = extract_s2_zip(s2_zip_path, EXTRACT_DIR)
        safe_name = os.path.basename(safe_root).replace('.SAFE', '')
        s2_stack  = os.path.join(EXTRACT_DIR, f'{safe_name}_S2_stack.tif')
        stack_s2_bands(safe_root, s2_stack)

        # Step 2: 씬 단위 Phase Correlation
        print('  [2/3] 씬 단위 Phase Correlation')
        dx_px, dy_px = estimate_scene_shift(k3a_ortho_path, s2_stack)
        print(f'         추정 shift: dx={dx_px:.3f}px dy={dy_px:.3f}px '
              f'({dx_px*K3A_RES:.2f}m, {dy_px*K3A_RES:.2f}m)')

        s2_stack_shifted = s2_stack.replace('_S2_stack.tif', '_S2_stack_shifted.tif')
        apply_scene_shift(s2_stack, s2_stack_shifted, dx_px, dy_px)

        # Step 3: 칩 생성
        print('  [3/3] 칩 생성')
        n_saved, n_drop = chip_pair_v9(
            k3a_ortho_path=k3a_ortho_path,
            s2_stack_path=s2_stack_shifted,
            chip_dir=CHIP_DIR,
            pair_name=k3a_folder,
            k3a_chip_sz=K3A_CHIP, s2_chip_sz=S2_CHIP,
            k3a_res=K3A_RES, s2_res=S2_RES,
            k3a_nodata_thresh=K3A_NODATA_THRESH,
            s2_nodata_thresh=S2_NODATA_THRESH,
        )
        results_log.append((k3a_folder, n_saved, n_drop, dx_px, dy_px))
        print(f'  ✅ {n_saved}칩 저장  제거={n_drop}칩')

    except Exception as e:
        print(f'  ❌ 오류: {e}')
        results_log.append((k3a_folder, 0, -1, 0.0, 0.0))

print(f"\n{'='*60}\n🏁 전체 처리 완료")

In [ ]:
from rasterio.warp import transform_bounds

with rasterio.open(k3a_ortho_path) as src:
    print('K3A bounds:', src.bounds)
    print('K3A CRS:', src.crs)

with rasterio.open(s2_stack) as src:
    print('S2  bounds:', src.bounds)
    print('S2  CRS:', src.crs)

# K3A bounds를 S2 CRS로 변환해서 비교
with rasterio.open(k3a_ortho_path) as k3a_src, rasterio.open(s2_stack) as s2_src:
    k3a_in_s2 = transform_bounds(k3a_src.crs, s2_src.crs, *k3a_src.bounds)
    print('\nK3A bounds (S2 CRS 변환):', k3a_in_s2)
    print('S2  bounds:', s2_src.bounds)

## 3. 최종 결과 요약

In [ ]:
total_saved = sum(r[1] for r in results_log)
total_drop  = sum(r[2] for r in results_log if r[2] >= 0)
total_error = sum(1 for r in results_log if r[2] == -1)

print('='*70)
print('최종 칩 생성 요약')
print('='*70)
print(f'처리 씬  : {len(results_log):>4}개')
print(f'저장 칩  : {total_saved:>5}개')
print(f'제거 칩  : {total_drop:>5}개  (nodata 초과)')
print(f'오류 씬  : {total_error:>4}개')
print()
print(f"{'씬명':<45} {'저장':>5} {'제거':>5} {'dx(px)':>8} {'dy(px)':>8} {'dist(m)':>8}")
print('-'*80)
for name, ns, nd, dx, dy in results_log:
    nd_str   = str(nd) if nd >= 0 else 'ERR'
    dist_m   = (dx**2 + dy**2)**0.5 * K3A_RES if nd >= 0 else 0
    print(f'{name:<45} {ns:>5} {nd_str:>5} {dx:>8.3f} {dy:>8.3f} {dist_m:>8.2f}')


# 디버깅


In [ ]:
import glob, os
import rasterio

stack_files = sorted(glob.glob(os.path.join(EXTRACT_DIR, '*_S2_stack.tif')))
print(f'전체 스택 파일: {len(stack_files)}개')

broken = []
for path in stack_files:
    try:
        with rasterio.open(path) as src:
            src.read(1, window=rasterio.windows.Window(0, 0, 10, 10))
        print(f'  ✅ {os.path.basename(path)[:60]}')
    except Exception as e:
        print(f'  ❌ {os.path.basename(path)[:60]}')
        broken.append(path)

print(f'\n손상된 파일: {len(broken)}개')
for p in broken:
    print(f'  {os.path.basename(p)}')

# 31개 씬 각각에 대응하는 S2 스택이 있는지 확인
missing_stacks = []

for scene in ortho_result_map:
    zips = glob.glob(os.path.join(DOWNLOAD_DIR, f'{scene}_*MSIL1C*.zip'))
    if not zips:
        zips = glob.glob(os.path.join(DOWNLOAD_DIR, f'{scene}_*MSIL2A*.zip'))
    if not zips:
        print(f'  ❌ zip 없음: {scene[-20:]}')
        continue

    with zipfile.ZipFile(zips[0], 'r') as z:
        safe_name = next((n.split('/')[0] for n in z.namelist() if '.SAFE' in n), None)

    safe_name_clean = safe_name.replace('.SAFE', '')
    s2_stack = os.path.join(EXTRACT_DIR, f'{safe_name_clean}_S2_stack.tif')

    if not os.path.exists(s2_stack):
        missing_stacks.append((scene, s2_stack))

print(f'스택 없는 씬: {len(missing_stacks)}개')
for scene, stack in missing_stacks[:5]:
    print(f'  {scene[-20:]}: {os.path.basename(stack)}')

In [ ]:
import os

# 손상된 스택 삭제 후 재생성
s2_stack = os.path.join(EXTRACT_DIR,
    'S2A_MSIL1C_20170304T021601_N0500_R003_T52SCG_20230925T201530_S2_stack.tif')

if os.path.exists(s2_stack):
    os.remove(s2_stack)
    print('삭제 완료')

# 10582 씬의 L1C zip으로 재생성
scene   = 'K3A_20170223044142_10582_00004060_L1R'
l1c_zip = glob.glob(os.path.join(DOWNLOAD_DIR, f'{scene}_*MSIL1C*.zip'))[0]
print(f'zip: {os.path.basename(l1c_zip)}')

safe_root = extract_s2_zip(l1c_zip, EXTRACT_DIR)
stack_s2_bands(safe_root, s2_stack)

# 확인
import rasterio
with rasterio.open(s2_stack) as src:
    r = src.read(1).astype('float32')
    print(f'max={r.max():.0f}  0값비율={(r==0).mean():.1%}')

In [ ]:
with rasterio.open(s2_stack) as src:
    print(f'밴드 수: {src.count}')
    for b in range(1, src.count+1):
        arr = src.read(b).astype(np.float32)
        print(f'  Band {b}: max={arr.max():.0f}  >0비율={(arr>0).mean():.1%}')